In [4]:
import os
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from urllib.parse import quote_plus
from dotenv import load_dotenv
from faker import Faker
from sqlalchemy import create_engine
import pymysql

# -----------------------------------------------------------------------------
# 1. Reproducibility & Database Connection Setup
# -----------------------------------------------------------------------------
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

load_dotenv(override=True)

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1').strip()
DB_PORT = os.getenv('DB_PORT', '3306').strip()
DB_NAME = os.getenv('DB_NAME')

assert DB_PASSWORD is not None, "Error: DB_PASSWORD is missing in .env file"

encoded_password = quote_plus(DB_PASSWORD)
connection_string = f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string, pool_pre_ping=True)

# Fetch customer IDs from real Olist table loaded in Step 1
print("Fetching customer IDs from MySQL...")
olist_customers = pd.read_sql("SELECT DISTINCT customer_unique_id FROM customers;", con=engine)
customer_pool = olist_customers['customer_unique_id'].tolist()
print(f"✅ Successfully fetched {len(customer_pool):,} customer IDs from Olist MySQL database!")

# -----------------------------------------------------------------------------
# 2. Generate User Profiles & A/B Group Assignments
# -----------------------------------------------------------------------------
acquisition_channels = ['Organic Search', 'Paid Search', 'Social Media', 'Direct', 'Email']
channel_weights = [0.35, 0.25, 0.20, 0.10, 0.10]

user_records = []
ab_assignment_records = []

start_date = datetime(2017, 1, 1)
end_date = datetime(2018, 6, 30)

for user_id in customer_pool:
    signup_time = fake.date_time_between(start_date=start_date, end_date=end_date)
    channel = np.random.choice(acquisition_channels, p=channel_weights)
    ab_group = 'treatment' if random.random() > 0.5 else 'control'
    
    user_records.append({
        'user_id': user_id,
        'signup_timestamp': signup_time,
        'acquisition_channel': channel
    })
    
    ab_assignment_records.append({
        'user_id': user_id,
        'group_assignment': ab_group
    })

users_df = pd.DataFrame(user_records)
ab_assignments_df = pd.DataFrame(ab_assignment_records)

print(f"Users generated: {len(users_df):,}")

# -----------------------------------------------------------------------------
# 3. Simulate Funnel Events Stream (events Table)
# -----------------------------------------------------------------------------
event_records = []
ab_outcomes = []

ACTIVATION_RATE = 0.60
CONTROL_PURCHASE_RATE = 0.150
TREATMENT_PURCHASE_RATE = 0.165

group_lookup = dict(zip(ab_assignments_df['user_id'], ab_assignments_df['group_assignment']))

for idx, row in users_df.iterrows():
    u_id = row['user_id']
    t_signup = row['signup_timestamp']
    group = group_lookup[u_id]
    
    # Event 1: Signup
    event_records.append({
        'user_id': u_id,
        'event_type': 'signup',
        'event_timestamp': t_signup,
        'acquisition_channel': row['acquisition_channel']
    })
    
    # Event 2: Activation
    is_activated = random.random() < ACTIVATION_RATE
    converted_to_purchase = 0
    
    if is_activated:
        t_activated = t_signup + timedelta(minutes=random.randint(5, 120))
        event_records.append({
            'user_id': u_id,
            'event_type': 'activated',
            'event_timestamp': t_activated,
            'acquisition_channel': row['acquisition_channel']
        })
        
        # Event 3: Purchase
        p_threshold = TREATMENT_PURCHASE_RATE if group == 'treatment' else CONTROL_PURCHASE_RATE
        is_purchased = random.random() < p_threshold
        
        if is_purchased:
            converted_to_purchase = 1
            t_purchased = t_activated + timedelta(hours=random.randint(1, 48))
            event_records.append({
                'user_id': u_id,
                'event_type': 'purchased',
                'event_timestamp': t_purchased,
                'acquisition_channel': row['acquisition_channel']
            })
            
    ab_outcomes.append(converted_to_purchase)

ab_assignments_df['converted'] = ab_outcomes
events_df = pd.DataFrame(event_records)

print(f"Total Events Generated: {len(events_df):,}")

# -----------------------------------------------------------------------------
# 4. Export Synthetic Tables to MySQL
# -----------------------------------------------------------------------------
print("\nExporting synthetic tables to MySQL `ecommerce_db`...")

events_df.to_sql('events', con=engine, if_exists='replace', index=False)
ab_assignments_df.to_sql('ab_test_assignment', con=engine, if_exists='replace', index=False)

print("🚀 Step 2 Complete: `events` and `ab_test_assignment` successfully loaded into MySQL!")

Fetching customer IDs from MySQL...
✅ Successfully fetched 96,096 customer IDs from Olist MySQL database!
Users generated: 96,096
Total Events Generated: 162,680

Exporting synthetic tables to MySQL `ecommerce_db`...
🚀 Step 2 Complete: `events` and `ab_test_assignment` successfully loaded into MySQL!
